# Migrant Archive — Colab Transcription

Transcribe a YouTube video using **faster-whisper large-v3** on Colab's free T4 GPU.

### What this notebook does
1. Clones the project repo (or uses uploaded files)
2. Downloads audio from YouTube via `yt-dlp`
3. Transcribes using `faster-whisper` (large-v3, GPU)
4. Saves the result as JSON to your Google Drive

### Estimated time
| Video length | Transcription time on T4 GPU |
|-------------|------------------------------|
| 5 min | ~15 seconds |
| 30 min | ~2 minutes |
| 2 hours | ~8-10 minutes |

### After this notebook
Download the JSON from Drive → place in `data/raw/whisper/` → run `rag_test.py --rebuild`

## 1. Install Dependencies

In [1]:
!pip install -q yt-dlp faster-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 119.3 MB/s eta 0:00:00


## 2. Mount Google Drive

This is where the JSON output will be saved. You'll need to authorize access.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/migrant-archive"
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/audio", exist_ok=True)

print(f"Drive mounted. Output → {DRIVE_ROOT}/output/")

Mounted at /content/drive
Drive mounted. Output → /content/drive/MyDrive/migrant-archive/output/


## 3. Get the Ingestion Code

Choose **one** method below. Option A (git clone) is recommended — it pulls all files with correct imports.

### Option A: Clone the repo (recommended)

In [ ]:
#import sys
from pathlib import Path

# Clone the repo
REPO_URL = "https://github.com/Sebastianlopez-dev/migrant-archive.git"
!git clone {REPO_URL} /content/migrant-archive

# Add backend/core to Python path
sys.path.insert(0, str(Path("/content/migrant-archive/backend/core")))

print("Repo cloned and path configured")

### Option B: Upload files manually

If you prefer not to clone, upload these 3 files from `backend/core/`:
- `ingestion.py`
- `ingestion_audio.py`  
- `ingestion_colab.py`

Use the Files panel (left sidebar) to upload, then run this cell:

In [4]:
# Run if using Option B (manual upload)
import sys
from pathlib import Path
sys.path.insert(0, "/content")
print("Manual upload path configured")

Manual upload path configured


## 4. Configure Your Video

Set the YouTube URL and language below. For the FILMIG channel, use `es`.
For Catalan/Spanish code-switching, try `ca` or `es`.

In [5]:
# ══════════════════════════════════════════
# EDIT THESE TWO VALUES
# ══════════════════════════════════════════

#VIDEO_URL = "https://www.youtube.com/watch?v=PASTE_VIDEO_ID_HERE"
VIDEO_URL = "https://www.youtube.com/watch?v=myxPJCDedOE"
LANGUAGE  = "es"       # es, en, ca, or "" for auto-detect

# ══════════════════════════════════════════
# Advanced (usually leave as-is)
# ══════════════════════════════════════════

MODEL     = "large-v3"  # tiny, base, small, medium, large-v3
DEVICE    = "cuda"      # cuda (GPU) or cpu

print(f"Video: {VIDEO_URL}")
print(f"Language: {LANGUAGE}  |  Model: {MODEL}  |  Device: {DEVICE}")

Video: https://www.youtube.com/watch?v=myxPJCDedOE
Language: es  |  Model: large-v3  |  Device: cuda


## 5. Run Transcription

This cell downloads the audio, runs Whisper, and saves the result.
For a 2-hour video, expect ~8-10 minutes on the free T4 GPU.

### 5.1 YouTube Authentication (cookies)

YouTube may block Colab's datacenter IP and ask for authentication.
If you see `Sign in to confirm you're not a bot`, follow these steps.

#### On your local machine (Mac terminal)
```bash
# Export cookies from Brave. Use 'chrome' if you use Chrome.
yt-dlp --cookies-from-browser brave "https://www.youtube.com" \
       --cookies cookies.txt --skip-download
```

In Colab
1. Upload cookies.txt using the Files panel (left sidebar — folder icon)
2. Run the cell below to inject cookies into all yt-dlp calls
3. Then run Section 6 (Run Transcription) as normal

> Skip this section if you don't get a bot-detection error.
If the video downloads fine without cookies, you don't need this.

In [6]:
import ingestion
import yt_dlp
from pathlib import Path

COOKIE_FILE = "/content/cookies.txt"

# Replace _fetch_metadata with cookie-aware version
def _fetch_metadata_cookies(video_url: str) -> dict:
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "cookiefile": COOKIE_FILE,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        return ydl.extract_info(video_url, download=False)

# Replace _download_audio with cookie-aware version
def _download_audio_cookies(video_url: str, output_dir: str = "data/audio") -> Path:
    audio_dir = Path(output_dir)
    audio_dir.mkdir(parents=True, exist_ok=True)

    info = _fetch_metadata_cookies(video_url)
    audio_path = audio_dir / f"{info['id']}.mp3"

    if audio_path.exists():
        return audio_path

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(audio_dir / "%(id)s.%(ext)s"),
        "noplaylist": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
        "quiet": True,
        "no_warnings": True,
        "cookiefile": COOKIE_FILE,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.extract_info(video_url, download=True)

    return audio_path

# Monkeypatch the module
ingestion._fetch_metadata = _fetch_metadata_cookies
ingestion._download_audio = _download_audio_cookies

print("Cookies injected — yt-dlp will authenticate as you")

Cookies injected — yt-dlp will authenticate as you


## 5.2 Extraction

In [7]:
import time
import json
from ingestion import VideoData, _fetch_metadata, _build_videodata, _download_audio
from ingestion_audio import _detect_device, _compute_type_for, _transcribe_audio

# ── Step 1: Fetch metadata ────────────────────────────────────────────
print("Fetching video metadata ...")
info = _fetch_metadata(VIDEO_URL)
video_id = info["id"]
title = info.get("title", "Unknown")
duration_mins = info.get("duration", 0) / 60

print(f"   Title: {title}")
print(f"   Duration: {duration_mins:.0f} min")
print(f"   Channel: {info.get('channel', 'Unknown')}")

# ── Step 2: Download audio ────────────────────────────────────────────
print(f"\n⬇ Downloading audio ...")
t0 = time.time()
audio_path = _download_audio(VIDEO_URL, output_dir=f"{DRIVE_ROOT}/audio")
print(f"   Done in {time.time() - t0:.0f}s → {audio_path}")

# ── Step 3: Transcribe ────────────────────────────────────────────────
print(f"\nTranscribing with faster-whisper ({MODEL}, {DEVICE}) ...")
print(f"    This may take a few minutes for long videos ...")
t0 = time.time()

segments = _transcribe_audio(
    audio_path=str(audio_path),
    language=LANGUAGE,
    model_size=MODEL,
    device=DEVICE,
)

elapsed = time.time() - t0
rtf = elapsed / (info.get("duration", 1) / 60)  # real-time factor
print(f"   Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {rtf:.1f}x real-time")
print(f"   Segments: {len(segments)}")

# ── Step 4: Build VideoData & save ────────────────────────────────────
video_data = _build_videodata(info, segments)
output_path = video_data.save_json(output_dir=f"{DRIVE_ROOT}/output")

print(f"\nSaved to Drive: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.0f} KB")

# ── Step 5: Preview ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"PREVIEW (first 300 chars):")
print(f"{'─'*60}")
print(video_data.full_text[:100])
print(f"{'─'*60}")

# ── Step 6: Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TRANSCRIPTION COMPLETE")
print(f"{'='*60}")
print(f"Video:    {title}")
print(f"Duration: {duration_mins:.0f} min")
print(f"Segments: {len(segments)}")
print(f"Chars:    {len(video_data.full_text):,}")
print(f"Speed:    {rtf:.1f}x real-time")
print(f"Saved:    {output_path}")
print(f"\n Next: Download this JSON → place in data/raw/whisper/ → rebuild index")

Fetching video metadata ...


   Title: Conversatorio: Estética y política en la literatura palestina contemporánea | FILMIG 2024
   Duration: 109 min
   Channel: Plataforma Cero

⬇ Downloading audio ...


   Done in 170s → /content/drive/MyDrive/migrant-archive/audio/myxPJCDedOE.mp3

Transcribing with faster-whisper (large-v3, cuda) ...
    This may take a few minutes for long videos ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   Done in 737s (12.3 min)
   Speed: 6.8x real-time
   Segments: 2491

Saved to Drive: /content/drive/MyDrive/migrant-archive/output/myxPJCDedOE.json
   File size: 979 KB

────────────────────────────────────────────────────────────
PREVIEW (first 300 chars):
────────────────────────────────────────────────────────────
Buenas tardes a todas, todos y todes. Bienvenidas, bienvenidos y bienvenides a el pistoletazo de sal
────────────────────────────────────────────────────────────

TRANSCRIPTION COMPLETE
Video:    Conversatorio: Estética y política en la literatura palestina contemporánea | FILMIG 2024
Duration: 109 min
Segments: 2491
Chars:    76,636
Speed:    6.8x real-time
Saved:    /content/drive/MyDrive/migrant-archive/output/myxPJCDedOE.json

 Next: Download this JSON → place in data/raw/whisper/ → rebuild index


## 6. Download the Result

The JSON is saved in your Google Drive at `migrant-archive/output/{video_id}.json`.

**Option A**: Download directly from [drive.google.com](https://drive.google.com) → My Drive → migrant-archive → output

**Option B**: Run this cell to download via Colab:

In [8]:
from google.colab import files

# Download the JSON file to your local machine
files.download(str(output_path))

print("\nPlace this file in: migrant-archive/data/raw/whisper/")
print("Then run: python backend/scripts/rag_test.py --rebuild")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Place this file in: migrant-archive/data/raw/whisper/
Then run: python backend/scripts/rag_test.py --rebuild


---

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `yt-dlp` says video unavailable | Video may be private/age-restricted. Try another URL. |
| Out of memory (OOM) | Switch to `MODEL = "medium"` — still excellent quality, less RAM. |
| GPU not available | Colab T4 quota exhausted. Wait a few hours or use `DEVICE = "cpu"` (much slower). |
| `faster-whisper` import error | Re-run the install cell (!pip install). Colab sometimes loses packages. |
| Drive mount fails | Re-run the mount cell. Make sure you're logged into the correct Google account. |